# ThermoNO on IC-ThermBench, FIXED orientation, v2 S4->S5 with a 300-epoch cap (follow-up to Sec 9.33)

The public IC-ThermBench release stores power / grid / k transposed in-plane relative to the temperature. Sec 9.29 was
produced with the as-stored orientation. This notebook reruns the same protocol (their split, their metrics, val-only
selection, S5 zero-shot from the frozen S4 model, same hyper-parameters and seeds) with `--fix-orientation` only.

Attach private dataset `rajulkabir/ic-thermbench-s2-s5`. Internet on. GPU T4 x1. Code: branch `iceforge`.

Only v2 S4->S5, seeds 0,1,2, epoch cap raised 150 -> 300 (patience 20, lr 2e-3 unchanged): the 150-epoch fixed runs hit the cap.


In [ ]:
import subprocess, sys, torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pyyaml'], check=True)

In [ ]:
import shutil, zipfile
from pathlib import Path
REPO = 'https://github.com/rajul-kk/thermo-3dic-surrogates.git'
CLONE, WORK = Path('/kaggle/working/repo'), Path('/kaggle/working')
OUT = WORK / 'icb_fixed300_results'; OUT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(CLONE, ignore_errors=True)
r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'iceforge', REPO, str(CLONE)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print('repo at', subprocess.run(['git', '-C', str(CLONE), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())

# Kaggle may or may not unpack the uploaded zip; handle both.
mats = list(Path('/kaggle/input').rglob('level2_steady/input.mat'))
if mats:
    DATA = mats[0].parent.parent
else:
    z = next(Path('/kaggle/input').rglob('s2_s5_datasets.zip'))
    zipfile.ZipFile(z).extractall(WORK / 'icb')
    DATA = next((WORK / 'icb').rglob('level2_steady/input.mat')).parent.parent
print('data root', DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
# (tag, scope, seed, extra args). Most important first: a timeout still leaves the headline.
S5 = ['--transfer', 'level5']
V1 = []   # script defaults = v1: lr 5e-3, 100 epochs, patience 15
V2 = ['--epochs', '300', '--patience', '20', '--lr', '2e-3']
RUNS = [('v2e300', 'level4', 0, V2 + S5), ('v2e300', 'level4', 1, V2 + S5), ('v2e300', 'level4', 2, V2 + S5)]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
import time
for tag, scope, seed, more in RUNS:
    out = OUT / f'icb_thermono_{scope}_{tag}_seed{seed}_fixed.json'
    if out.exists():
        print(scope, tag, seed, 'already done'); continue
    t = time.time()
    cmd = [sys.executable, 'scripts/icb_thermono.py', '--scope', scope, '--seed', str(seed), '--data-root', str(DATA),
           '--device', DEVICE, '--out', str(out), '--fix-orientation'] + more
    print('===', tag, scope, seed, ' '.join(more), flush=True)
    p = subprocess.Popen(cmd, cwd=CLONE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line:
            print('  ' + line, end='', flush=True)
    if p.wait() != 0:
        print(f'  !! {scope} {tag} {seed} failed, continuing'); continue
    print(f'  ({(time.time() - t) / 60:.1f} min)', flush=True)


In [ ]:
import json
for tag, scope, seed, more in RUNS:
    f = OUT / f'icb_thermono_{scope}_{tag}_seed{seed}_fixed.json'
    if not f.exists(): continue
    r = json.loads(f.read_text())
    s5 = r.get('level5', {})
    print(f"{tag:<4} {scope} seed{seed}: RMSE {r[scope]['rmse']:.4f} peak {r[scope]['max_temperature_error']:.3f} "
          f"S5 {s5.get('rmse', float('nan')):.4f} epochs {r['epochs_run']} ({r['minutes']:.0f} min)")
print('150-epoch fixed refs: v2 S4 3.15 +- 0.07 (3.21, 3.14, 3.08); S5 7.30 +- 0.51 (6.78, 7.31, 7.81); as stored S4 4.40 S5 6.70')
